# Ingest H&M data into Shaped

This example will show you how to prepare the H&M dataset ([link to Kaggle](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/overview)) and upload it to a Shaped table. 

# 1. Data preparation

# 1.1 Set up virtual environment and install dependencies

Create the venv with python 3.11 to ensure compatibility with the Shaped CLI:

```bash
python3.11 -m venv .venv
/.venv/bin/activate
```

In [ ]:
%pip install -qU shaped pandas

: 

# 1.2 Prepare datasets (items)

This example involves three datasets: 
- `articles.csv`: The catalog items, which will be candidates for our retrieval engine
- `customers.csv`: Information about each user; 
- `transaction_train.csv`: List of customer interactions and transactions; we'll use this to train our engine on behavioural data

Before uploading to Shaped, we have to ensure: 
1. Column names are only alphanumeric with underscores (no hyphens or special characters)
2. Dates are in epoch or ISO time

In [ ]:
import pandas as pd

data_dir = "data/raw"
articles_file = f"{data_dir}/articles.csv"
customers_file = f"{data_dir}/customers.csv"
transactions_file = f"{data_dir}/transactions_train.csv"

try:
    articles = pd.read_csv(articles_file)
    customers = pd.read_csv(customers_file)
    transactions = pd.read_csv(transactions_file)
    print('Dataframes loaded successfully')
except Exception:
    print('Error loading dataframes -' + Exception)


In [ ]:
print('#'*20 + ' Summary of data ' + '#'*20 + '\n')
print('#'*20 + ' ARTICLES DF ' + '#'*20)
print(articles.dtypes)
print('\n'+'#'*20 + ' CUSTOMERS DF ' + '#'*20)
print(customers.dtypes)
print('\n'+'#'*20 + ' TRANSACTIONS DF ' + '#'*20)
print(transactions.dtypes)

In [ ]:
from datetime import datetime
# articles needs no changes

# customers needs "FN" to be renamed "subscribed_to_fn" and Active should be lowercase
customers = customers.rename(columns={'FN': 'subscribed_to_fn', 'Active': 'active'})

# transactions needs t_date to be an epoch date (in ms)
transactions['created_at'] = (pd.to_datetime(transactions['t_dat']).view('int64') // 10**9).astype('int64')

print('#'*20 + ' Data cleaning steps completed ' + '#'*20)

## 1.3 Export dataframes as jsonl files 

Our datasets are structured correctly, so now it's time to upload them to Shaped. We can do this using the CLI:

In [ ]:
print("Exporting dataframes to JSONL files in 'data/processed/' directory...")
try:
    customers.to_json('data/processed/customers.jsonl', orient='records', lines=True)
    print("Customers df exported...")
    articles.to_json('data/processed/articles.jsonl', orient='records', lines=True)
    print("Articles df exported...")
except Exception:
    print(f'An error occurred: {Exception}')


In [ ]:
transactions.to_json('data/processed/transactions.jsonl', orient='records', lines=True)
print("Transactions df exported...")

# 1.4 Upload data to Shaped

Use the CLI to upload each dataset to Shaped:

```bash
shaped create-dataset-from-uri --name hm__articles --type jsonl --path data/processed/articles.jsonl
shaped create-dataset-from-uri --name hm__customers --type jsonl --path data/processed/customers.jsonl
shaped create-dataset-from-uri --name hm__transactions --type jsonl --path data/processed/transactions.jsonl
```